In [1]:
import pandas as pd
pd.options.plotting.backend = "plotly"

from summer2 import CompartmentalModel
from summer2.parameters import Parameter

We build on the previous model, the SEIR model with clearance, and introduce waning immunity. We just add a flow from recovered to susceptible - ie. for now we don't track people's immunity status.

In [ ]:
def build_seirs_model(
        config: dict,
) -> CompartmentalModel:
    """ 
    Args: 
        config: dict with configs other than params
    Returns:
        model: the summer model obejct
    """

    # For now we just keep the same compartments, ie. we don't define a new susceptible compartment for recovered people
    compartments = (
        "susceptible",
        "exposed",
        "infectious",
        "recovered",
    )
    analysis_times = (config["start_time"], config["end_time"])

    model = CompartmentalModel(
        times = analysis_times,
        compartments = compartments,
        infectious_compartments = ("infectious",),
    )

    model.set_initial_population(
        distribution={
            "susceptible": config["population"] - config["seed"],
            "infectious": config["seed"],
        },
    )
    
    model.add_crude_birth_flow(
        "births",
        Parameter("birth_rate"),
        "susceptible",
    )

    model.add_universal_death_flows(
        "background_mortality",
        death_rate=Parameter("death_rate",)
    )

    model.add_infection_frequency_flow(
        name="infection",
        contact_rate=Parameter("contact_rate"),
        source="susceptible",
        dest="exposed",
    )
    model.add_transition_flow(
        name="progression",
        fractional_rate=Parameter("progression"),
        source="exposed",
        dest="infectious",
    )
    model.add_transition_flow(
        name="recovery",
        fractional_rate=Parameter("recovery"),
        source="infectious",
        dest="recovered",
    )

    model.add_transition_flow(
        name="natural_clearance",
        fractional_rate=Parameter("clearance_rate"),
        source="exposed",
        dest="recovered",
    )

    model.add_transition_flow(
        name="waning_immunity",
        fractional_rate=Parameter("waning_immunity"),
        source="recovered",
        dest="susceptible"                
    )

    return model

In [ ]:
# We use the same configurations:
model_config = {
    "population": 48928.0, # TB study pop in 2004 in Guinea-Bissau, we assume everyone is susceptible if not infected
    "seed": 144.0, # Number of cases diagnosed in 2004 - the seed
    "start_time": 2004,
    "end_time": 2020,
}

# Create instance of SEIRS model with these configs
seirs_model = build_seirs_model(model_config)

In [4]:
# But we add waning immunity. One study found that, on average, the rate of waning immunity was 57% over a ten year period.
# If we aasume that:
# r = (1 + R)^(1/n) - 1
# Where
# r is a constant 1-year rate over the 10 years.
# R is the cumulative 10 year rate, here 0.57
# n is the number of years, here 10
# Then r = (1 + 0.50)^(1/10) = 1.50^0.10 - 1 = 0.0414
parameters = {
    "recovery": 0.15,
    "contact_rate": 10.0,
    "death_rate": 0.022, # Average life expectancy was 45.5 in 2004
    "birth_rate": 0.022, # We start by matching the death rate to keep the population stable over time

    "clearance_rate": 0.8, # Say 80% of infected peolpe actually recover naturally
    "progression": 0.2, # Should then be 20% to give the average dwell time in the latent compartment of 1 year
    "waning_immunity": 0.0414
}

In [5]:
# Run and plot:
seirs_model.run(parameters=parameters)
compartment_values = seirs_model.get_outputs_df()
compartment_values.plot(
    labels={"index": "time", "value": "compartment size"}
)

We're getting closer to dynamics that agree with data from Guinea-Bissau (although this doesn't necessarily mean that the model has anything to do with biological reality!). For many years, the incidence of TB in the country has been relatively stable, and the model is starting to reflect this pattern. However, the numbers are still catastrophicly high, and they do not reflect the situation of TB in any period or place in history to my knowledge. The next thing we need to take into account to get a more realistic picture of the true TB dynamics is that TB increases mortality. Right now, the mortality in the infectious (disease) compartment is the same as in any other compartment. TB increases mortality in many ways. One is the mortality arising from people that die in their acute disease stage. Another is the generally increased mortality among people who have had tuberculosis but are recovered. This introduces complexity to the model and our understanding of TB, because we need the model to remember who have had TB in the past. The people who recover from infection without getting the disease are different from the people wgo get the disease. The people who have TB disease have a higher direct/acute mortality rate, but they are also suffering from higher degrees of morbidity in the rest of their lives. 

In the next notebook, we introduce the increased mortality among people actively suffering from TB disease. In our model, we don't have any screening or treatment, so we stick to older estimates of mortality before antibiotics. 
